In [1]:
# ============================================================
# GenAI Text Summarizer using BART + Gradio + ROUGE Evaluation
# ============================================================

# 1. Install required libraries
!pip install -q -U transformers gradio evaluate rouge_score sentencepiece

# ============================================================
# 2. Import Libraries
# ============================================================

import gradio as gr
import evaluate
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ============================================================
# 3. Load BART Summarization Model
# ============================================================

model_name = "facebook/bart-large-cnn"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")


# ============================================================
# 4. Summarization Function
# ============================================================

def summarize_text(input_text):

    if not input_text.strip():
        return "Please enter some text to summarize."

    # Tokenize input text
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    # Generate summary
    with torch.no_grad():

        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=45,
            min_length=15,
            num_beams=4,
            do_sample=False
        )

    # Convert generated tokens into text
    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


# ============================================================
# 5. Create Gradio Application
# ============================================================

demo = gr.Interface(
    fn=summarize_text,

    inputs=gr.Textbox(
        lines=8,
        label="Enter text to summarize",
        placeholder="Enter a paragraph here..."
    ),

    outputs=gr.Textbox(
        label="Generated Summary"
    ),

    title="GenAI Text Summarizer",

    description=(
        "A Generative AI text summarization application "
        "using BART, Transformers and Gradio."
    )
)


# ============================================================
# 6. ROUGE Evaluation
# ============================================================

print("\nLoading ROUGE evaluation...")

rouge = evaluate.load("rouge")

generated_summaries = [
    "AI models generate new content such as text and images."
]

reference_summaries = [
    "Generative AI models are capable of producing new content including text and images."
]

scores = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

print("\nROUGE Evaluation Scores:")
print("--------------------------------")

for metric, score in scores.items():
    print(f"{metric}: {score:.4f}")


# ============================================================
# 7. Launch Gradio Application
# ============================================================

print("\nLaunching Gradio application...")

demo.launch(share=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
Loading model...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully!

Loading ROUGE evaluation...



ROUGE Evaluation Scores:
--------------------------------
rouge1: 0.6087
rouge2: 0.3810
rougeL: 0.6087
rougeLsum: 0.6087

Launching Gradio application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f36460538914420c6e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
